In [2]:
#virtual environment setup code
import os
os.chdir(r"C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2")
print(os.getcwd())

C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2


In [3]:
# Install necessary packages
!pip install pandas nltk
import pandas as pd
import nltk #natural language toolkit
nltk.download('punkt') #tokenizer data
nltk.download('stopwords') #stopwords data 

[nltk_data] Downloading package punkt to
[nltk_data]     c:\Users\Prana\OneDrive\Documents\GitHub\infosys-
[nltk_data]     langgraph-email-assistant-group2\.venv\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     c:\Users\Prana\OneDrive\Documents\GitHub\infosys-
[nltk_data]     langgraph-email-assistant-group2\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
#loading the dataset
df = pd.read_csv("data/email_evaluation_dataset_pranaya.csv")
df.head()

,id,email_text,expected_action,expected_tone
0,1,You are receiving this email as part of our re...,ignore,neutral
1,2,"Dear Meera, just a quick reminder that we have...",respond,polite
2,3,This is our monthly newsletter highlighting re...,ignore,neutral
3,4,"Hello Meera, this is to inform you that your a...",notify,polite
4,5,"Hello Meera, this is a follow-up regarding the...",respond,urgent


In [7]:
import pandas as pd #data manipulation library
import numpy as np #numerical computing library
import re #regular expressions library

In [8]:
# Data Preprocessing: Cleaning the email body text
df['clean_text'] = (
    df['email_text']
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)
df[['email_text', 'clean_text']].head()

,email_text,clean_text
0,You are receiving this email as part of our re...,you are receiving this email as part of our re...
1,"Dear Meera, just a quick reminder that we have...",dear meera just a quick reminder that we have ...
2,This is our monthly newsletter highlighting re...,this is our monthly newsletter highlighting re...
3,"Hello Meera, this is to inform you that your a...",hello meera this is to inform you that your ac...
4,"Hello Meera, this is a follow-up regarding the...",hello meera this is a followup regarding the p...


In [9]:
# Keyword Extraction: Removing stop words
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words("english"))
df['keywords'] = df['clean_text'].apply(
    lambda x: [w for w in x.split() if w not in stop_words]
)
df[['clean_text', 'keywords']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     c:\Users\Prana\OneDrive\Documents\GitHub\infosys-
[nltk_data]     langgraph-email-assistant-group2\.venv\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,clean_text,keywords
0,you are receiving this email as part of our re...,"[receiving, email, part, regular, information,..."
1,dear meera just a quick reminder that we have ...,"[dear, meera, quick, reminder, meeting, regard..."
2,this is our monthly newsletter highlighting re...,"[monthly, newsletter, highlighting, recent, up..."
3,hello meera this is to inform you that your ac...,"[hello, meera, inform, academic, submission, d..."
4,hello meera this is a followup regarding the p...,"[hello, meera, followup, regarding, pending, i..."


In [11]:
# Triage Rule Implementation
def triage_rule(text):
    text = text.lower()
    if "refund" in text or "urgent" in text or "password" in text:
        return "notify_human"
    if "newsletter" in text or "promotion" in text:
        return "ignore"
    return "respond_or_act"
df['triage'] = df['clean_text'].apply(triage_rule)
df[['email_text', 'triage']].head()

,email_text,triage
0,You are receiving this email as part of our re...,respond_or_act
1,"Dear Meera, just a quick reminder that we have...",respond_or_act
2,This is our monthly newsletter highlighting re...,ignore
3,"Hello Meera, this is to inform you that your a...",respond_or_act
4,"Hello Meera, this is a follow-up regarding the...",respond_or_act


In [13]:
# 1. Imports + load
import pandas as pd
import re
from IPython.display import display
# adjust path if needed
CSV_PATH = "data/email_evaluation_dataset_pranaya.csv"
df = pd.read_csv(CSV_PATH)
print("Loaded rows:", len(df))
display(df.head())

Loaded rows: 100


,id,email_text,expected_action,expected_tone
0,1,You are receiving this email as part of our re...,ignore,neutral
1,2,"Dear Meera, just a quick reminder that we have...",respond,polite
2,3,This is our monthly newsletter highlighting re...,ignore,neutral
3,4,"Hello Meera, this is to inform you that your a...",notify,polite
4,5,"Hello Meera, this is a follow-up regarding the...",respond,urgent


In [15]:
# 2. Create a helpful cleaned text column
# Keep words and meaningful tokens (remove only repeated whitespace and control chars)
import re
def make_clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    # normalize spaces and lowercase
    s = re.sub(r'\s+', ' ', s)
    s = s.lower()
    return s
source_col = 'email_text'
df['clean_text'] = df[source_col].apply(make_clean_text)
df[[source_col, 'clean_text']].head()

,email_text,clean_text
0,You are receiving this email as part of our re...,you are receiving this email as part of our re...
1,"Dear Meera, just a quick reminder that we have...","dear meera, just a quick reminder that we have..."
2,This is our monthly newsletter highlighting re...,this is our monthly newsletter highlighting re...
3,"Hello Meera, this is to inform you that your a...","hello meera, this is to inform you that your a..."
4,"Hello Meera, this is a follow-up regarding the...","hello meera, this is a follow-up regarding the..."


In [16]:
# 3. Debug-friendly triage (regex-based with priority). This returns (label, matched_pattern)
RULES = {
    'notify_human': [
        r'\binvoice\b', r'\bbill\b', r'\bpayment\b', r'\bpassword\b', r'\breset\b',
        r'\bfraud\b', r'\bchargeback\b', r'\bdispute\b', r'\bdenied\b', r'\boverdue\b'
    ],
    'respond_or_act': [
        r'\bmeeting\b', r'\bschedule\b', r'\bscheduled\b', r'\battached\b', r'\battachment\b',
        r'\bplease find\b', r'\bplease review\b', r'\baction required\b', r'\burgent\b', r'\basap\b'
    ],
    'ignore': [
        r'\bunsubscribe\b', r'\bpromotion\b', r'\bsale\b', r'\boffer\b', r'\bnewsletter\b',
        r'\bsurvey\b', r'\bcongratulations\b'
    ]
}
COMPILED = {k:[re.compile(p, flags=re.I) for p in v] for k,v in RULES.items()}
def triage_rule_debug(text):
    if not isinstance(text, str) or text.strip()=="":
        return ('ignore','empty_text')
    for label in ['notify_human','respond_or_act','ignore']:
        for pat in COMPILED[label]:
            if pat.search(text):
                return (label, pat.pattern)
    return ('respond_or_act','default_fallback')
# apply and store both label and matched pattern
df[['triage','triage_match']] = df['clean_text'].apply(lambda t: pd.Series(triage_rule_debug(t)))
print("Triage value counts (advanced):")
print(df['triage'].value_counts())
display(df[['clean_text','triage','triage_match']].head(20))

Triage value counts (advanced):
triage
respond_or_act    77
notify_human      22
ignore             1
Name: count, dtype: int64


,clean_text,triage,triage_match
0,you are receiving this email as part of our re...,respond_or_act,default_fallback
1,"dear meera, just a quick reminder that we have...",respond_or_act,\bmeeting\b
2,this is our monthly newsletter highlighting re...,ignore,\bnewsletter\b
3,"hello meera, this is to inform you that your a...",respond_or_act,default_fallback
4,"hello meera, this is a follow-up regarding the...",notify_human,\binvoice\b
5,"hello alex, this is to inform you that your ac...",respond_or_act,default_fallback
6,"hi ananya, this is a reminder about our projec...",respond_or_act,\bmeeting\b
7,"hi kiran, this is a reminder about our project...",respond_or_act,\bmeeting\b
8,"hi pranaya, you have been shortlisted for the ...",respond_or_act,default_fallback
9,here are the latest updates from our organizat...,respond_or_act,default_fallback


In [18]:
# 5. Diagnostics — use this in live demo to show interns why each label was chosen
print("Top 20 rows with triage reasons:")
display(df[['email_text','clean_text','triage','triage_match']].head(20))

# show a few sample rows where triage==notify_human for inspection
print("\nSample notify_human rows:")
display(df[df['triage']=='notify_human'][['email_text','clean_text','triage_match']].head(6))
print("\nSample respond_or_act rows:")
display(df[df['triage']=='respond_or_act'][['email_text','clean_text','triage_match']].head(6))

print("\nSample ignore rows:")
display(df[df['triage']=='ignore'][['email_text','clean_text','triage_match']].head(6))

Top 20 rows with triage reasons:


,email_text,clean_text,triage,triage_match
0,You are receiving this email as part of our re...,you are receiving this email as part of our re...,respond_or_act,default_fallback
1,"Dear Meera, just a quick reminder that we have...","dear meera, just a quick reminder that we have...",respond_or_act,\bmeeting\b
2,This is our monthly newsletter highlighting re...,this is our monthly newsletter highlighting re...,ignore,\bnewsletter\b
3,"Hello Meera, this is to inform you that your a...","hello meera, this is to inform you that your a...",respond_or_act,default_fallback
4,"Hello Meera, this is a follow-up regarding the...","hello meera, this is a follow-up regarding the...",notify_human,\binvoice\b
5,"Hello Alex, this is to inform you that your ac...","hello alex, this is to inform you that your ac...",respond_or_act,default_fallback
6,"Hi Ananya, this is a reminder about our projec...","hi ananya, this is a reminder about our projec...",respond_or_act,\bmeeting\b
7,"Hi Kiran, this is a reminder about our project...","hi kiran, this is a reminder about our project...",respond_or_act,\bmeeting\b
8,"Hi Pranaya, you have been shortlisted for the ...","hi pranaya, you have been shortlisted for the ...",respond_or_act,default_fallback
9,Here are the latest updates from our organizat...,here are the latest updates from our organizat...,respond_or_act,default_fallback



Sample notify_human rows:


,email_text,clean_text,triage_match
4,"Hello Meera, this is a follow-up regarding the...","hello meera, this is a follow-up regarding the...",\binvoice\b
15,"Dear Rohit, our records indicate that invoice ...","dear rohit, our records indicate that invoice ...",\binvoice\b
17,"Dear Ananya, our records indicate that invoice...","dear ananya, our records indicate that invoice...",\binvoice\b
31,"Dear Rohit, our records indicate that invoice ...","dear rohit, our records indicate that invoice ...",\binvoice\b
34,"Dear Aditi, our records indicate that invoice ...","dear aditi, our records indicate that invoice ...",\binvoice\b
35,"Dear Aditi, our records indicate that invoice ...","dear aditi, our records indicate that invoice ...",\binvoice\b



Sample respond_or_act rows:


,email_text,clean_text,triage_match
0,You are receiving this email as part of our re...,you are receiving this email as part of our re...,default_fallback
1,"Dear Meera, just a quick reminder that we have...","dear meera, just a quick reminder that we have...",\bmeeting\b
3,"Hello Meera, this is to inform you that your a...","hello meera, this is to inform you that your a...",default_fallback
5,"Hello Alex, this is to inform you that your ac...","hello alex, this is to inform you that your ac...",default_fallback
6,"Hi Ananya, this is a reminder about our projec...","hi ananya, this is a reminder about our projec...",\bmeeting\b
7,"Hi Kiran, this is a reminder about our project...","hi kiran, this is a reminder about our project...",\bmeeting\b



Sample ignore rows:


,email_text,clean_text,triage_match
2,This is our monthly newsletter highlighting re...,this is our monthly newsletter highlighting re...,\bnewsletter\b


In [19]:
df['predicted_triage'] = df['clean_text'].apply(triage_rule)
df[['email_text','predicted_triage']].head()

,email_text,predicted_triage
0,You are receiving this email as part of our re...,respond_or_act
1,"Dear Meera, just a quick reminder that we have...",respond_or_act
2,This is our monthly newsletter highlighting re...,ignore
3,"Hello Meera, this is to inform you that your a...",respond_or_act
4,"Hello Meera, this is a follow-up regarding the...",respond_or_act
